In [1]:
import os
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


def clear_screen():
    """Clears the terminal screen for better user experience."""
    os.system('cls' if os.name == 'nt' else 'clear')

def main_menu():
    """Displays the main menu and gets the user's choice."""
    clear_screen()
    print("Welcome to VegasIQ")
    print("===================")
    print("1. Las Vegas Events and Rooms Database")
    print("2. Las Vegas Weather Data")
    print("3. Las Vegas Airport Statistics")
    print("4. Clark County Meeting Space Inventory")
    print("5. User Management")
    print("6. System Exit")
    try:
        return int(input("Select an option: "))
    except ValueError:
        return -1  # Return an invalid option to trigger error handling.

def Events_Room_space():
    # Load the event and room data
    event_file_path = 'convention-calendar-2024-10-13_12-42-47.csv'
    events_df = pd.read_csv(event_file_path)
    room_file_path = 'Clark_County_Room_Inventory_Dec_2023.csv'
    hotels_df = pd.read_csv(room_file_path)

    # Data Cleaning and Conversion
    hotels_df['Rooms'] = hotels_df['Rooms'].str.replace(',', '').astype(str)
    hotels_df['Rooms'] = pd.to_numeric(hotels_df['Rooms'], errors='coerce').fillna(0)
    events_df['Start Date'] = pd.to_datetime(events_df['Start Date'], format='%m/%d/%Y')
    events_df['End Date'] = pd.to_datetime(events_df['End Date'], format='%m/%d/%Y')

    # Function to filter events

    def filter_events_dashboard(data, month=None, year=None, venue=None):
        if month:
            data = data[data['Start Date'].dt.month == month]
        if year:
            data = data[data['Start Date'].dt.year == year]
        if venue:
            data = data[data['Venue'].str.contains(venue, case=False, na=False)]
        return data[['Venue', 'Event', 'Start Date', 'End Date', 'Est Attendees']]

    # Function to display the events as a formatted table
    def display_pretty_table(data):
        if data.empty:
            print("No events found for the selected criteria.")
            return
        
        venue_width, event_width, date_width, attendees_width = 50, 60, 12, 15
        header = (f"{'Venue':<{venue_width}} {'Event':<{event_width}} {'Start Date':<{date_width}} "
                f"{'End Date':<{date_width}} {'Est Attendees':<{attendees_width}}")
        print(header)
        print("=" * len(header))
        for _, row in data.iterrows():
            venue = f"{row['Venue']:<{venue_width}}"[:venue_width]
            event = f"{row['Event']:<{event_width}}"[:event_width]
            start_date = row['Start Date'].strftime("%Y-%m-%d")
            end_date = row['End Date'].strftime("%Y-%m-%d")
            attendees = f"{row['Est Attendees']:<{attendees_width}}"
            print(f"{venue} {event} {start_date:<{date_width}} {end_date:<{date_width}} {attendees}")

    # Event Database Menu
    def events_database_menu():
        while True:
            print("\n-- Events Database --")
            print("1. Filter by Month")
            print("2. Filter by Year")
            print("3. Filter by Location (Venue)")
            print("4. View All Events")
            print("5. Return to Main Menu")
            choice = input("Select an option: ")
            if choice == '1':
                try:
                    month = int(input("Enter the month (1-12): "))
                    while True:
                        year = int(input("Enter the year (2024 or 2025): "))
                        if year in [2024, 2025]:
                            break
                        else:
                            print("No events found for the selected year, please try again")
                    print()  # Add a space between user input and the displayed results
                    filtered_events = filter_events_dashboard(events_df, month=month, year=year)
                    display_pretty_table(filtered_events)
                except ValueError:
                    print("Invalid input. Please enter a valid month and year.")
            elif choice == '2':
                try:
                    while True:
                        year = int(input("Enter the year (2024 or 2025): "))
                        if year in [2024, 2025]:
                            break
                        else:
                            print("No events found for the selected year, please try again")
                    filtered_events = filter_events_dashboard(events_df, year=year)
                    display_pretty_table(filtered_events)
                except ValueError:
                    print("Invalid input. Please enter a valid year.")
            elif choice == '3':
                while True:
                    venue = input("Enter the venue name (e.g., Las Vegas Convention Center): ").strip()
                    filtered_events = filter_events_dashboard(events_df, venue=venue)
                    if not filtered_events.empty:
                        break
                    else:
                        print(f"No events found for the venue: {venue}. Please try again.")
                display_pretty_table(filtered_events)
            elif choice == '4':
                display_pretty_table(filter_events_dashboard(events_df))
            elif choice == '5':
                main_menu()
                return
            else:
                print("Invalid option. Please try again.")

    # Event and Accommodation Report
    def event_and_accommodation_report():
        while True:
            venue_name = input("Enter the venue name: ")
            events_at_venue = events_df[events_df['Venue'].str.contains(venue_name, case=False, na=False, regex=True)]
            if events_at_venue.empty:
                print(f"No events found for the venue: {venue_name}. Please try again.")
            else:
                matched_hotels = hotels_df[hotels_df['Property Name'].str.contains(venue_name, case=False, na=False, regex=True)]
                if matched_hotels.empty:
                    print(f"No rooms found for the venue: {venue_name}")
                else:
                    for _, hotel_row in matched_hotels.iterrows():
                        print(f"\nHotel: {hotel_row['Property Name']}")
                        print(f"Total Available Rooms: {hotel_row['Rooms']}")
                for _, row in events_at_venue.iterrows():
                    print(f"\nEvent: {row['Event']}")
                    print(f"Venue: {row['Venue']}")
                    print(f"Start Date: {row['Start Date'].strftime('%Y-%m-%d')}")
                    print(f"End Date: {row['End Date'].strftime('%Y-%m-%d')}")
                    print(f"Estimated Attendees: {row['Est Attendees']}")
                break

    # Peak Attendance by Week
    def peak_attendance_scatter_plot():
        events_df['Week'] = events_df['Start Date'].dt.to_period('W')
        weekly_attendees = events_df.groupby('Week')['Est Attendees'].sum().reset_index()
        weekly_attendees['Total Rooms'] = hotels_df['Rooms'].sum()
        weekly_attendees['Week'] = weekly_attendees['Week'].dt.start_time
        x = weekly_attendees['Week']
        y = weekly_attendees['Est Attendees']
        colors = np.random.rand(len(weekly_attendees))
        sizes = 1000 * colors
        plt.figure(figsize=(16, 8))
        plt.scatter(x, y, c=colors, s=sizes, alpha=0.5, cmap='viridis')
        plt.axhline(y=hotels_df['Rooms'].sum(), color='r', linestyle='--', label='Total Available Rooms')
        plt.colorbar(label='Color Scale')
        plt.xlabel('Week')
        plt.ylabel('Estimated Attendees')
        plt.title('Peak Attendance by Week')
        plt.xticks(rotation=45, ha='right')
        plt.legend()
        plt.tight_layout()
        plt.show()

    # Sub Menu
    def lv_events_database_menu():
        while True:
            print("\n-------------   Events Database   ------------------")
            print(" Las Vegas Travel and Convention Intelligence")
            print(" ---------------------------------------------")
            print("1. Event Data")
            print("2. Event and Accommodation Report")
            print("3. Peak Attendance by Week")
            print("4. Return to Main Menu")
            choice = input("Enter your choice: ")
            if choice == '1':
                events_database_menu()
            elif choice == '2':
                event_and_accommodation_report()
            elif choice == '3':
                peak_attendance_scatter_plot()
            elif choice == '4':
                
                return

            else:
                print("Invalid option. Please try again.")
    
    lv_events_database_menu()



def airport_menu():
    while True:
        print("\n-- Airport Statistics --")
        print("1. Airport Statistics Yearly ")
        print("2. Return to Main Menu")
        choice = input("Select an option: ")
        # read the csv file for the data
        passengers = pd.read_csv('LV-Passengers.csv')
        flights = pd.read_csv('LV-Flights.csv')
        #Combine all data into one DataFrame
        transportation = pd.merge(passengers, flights, how = 'inner', on = ['Year','Month'])
        transportation
            
        # Data Cleaning:
        #Filter the TOTAL row 
        transportation = transportation.loc[transportation['Month'] != 'TOTAL']
        #Drop TOTAL PASSENGERS & TOTAL FLIGHTS column
        transportation = transportation.drop(['TOTAL PASSENGERS', 'TOTAL FLIGHTS'], axis = 1)
        #Remove commas and convert columns to numeric
        #Replace the NaN with median within a year
        dataCleaning = ['DOMESTIC PASSENGERS', 'INTERNATIONAL PASSENGERS', 
                            'DOMESTIC FLIGHTS', 'INTERNATIONAL FLIGHTS']
        for col in dataCleaning:
            transportation[col] = transportation[col].replace(',', '', regex=True).astype(float)
            transportation[col] = transportation.groupby('Year')[col].transform(lambda x: x.fillna(x.median()))
        if choice == '1':
            #Ask user to enter the year
            print('Please enter the year to review data (from 2002 to 2024)')
            year = input()
            
            #Verify the year in the DataFrame
            while not year.isdigit() or int(year) not in transportation['Year'].values:
                print('Currently, only the data form 2002 to 2024 are available, please enter again:')
                year = input()
            
            #Create the mask for filtering by year
            mask = transportation['Year'] == int(year)
            data = transportation[mask]
            data
            maskHistory = transportation['Year'] == int(year) - 1
            dataHistory = transportation[maskHistory]
            
            # Ensure both datasets have the same months
            uniqueMonths = data['Month'].unique()
            
            # Reindex both data and dataHistory to match months
            data = data.set_index('Month').reindex(uniqueMonths).reset_index()
            dataHistory = dataHistory.set_index('Month').reindex(uniqueMonths).reset_index()
            
            # Fill missing values in dataHistory with zeros
            dataHistory = dataHistory.fillna(0)
            
            # Side-by-side bar chart to compare current year with previous year
            if not dataHistory.empty:  # Check if there is data for the previous year
                plt.figure(figsize=(10, 6))
                width = 0.4  # Set Bar width
                x = range(len(data['Month']))
                # Plot the input year data
                plt.bar(data['Month'], data['DOMESTIC PASSENGERS'], width = width, label='Domestic '+str(year))
                plt.bar(data['Month'], data['INTERNATIONAL PASSENGERS'], width = width, 
                        bottom=data['DOMESTIC PASSENGERS'], label='International '+str(year))
                
                # Offset x-axis for the previous year
                xHistory = [i - width for i in x]
                
                # Plot the previous year data
                plt.bar(xHistory, dataHistory['DOMESTIC PASSENGERS'], width = width, label='Domestic '+str(int(year) - 1))
                plt.bar(xHistory, dataHistory['INTERNATIONAL PASSENGERS'], width = width, 
                        bottom=dataHistory['DOMESTIC PASSENGERS'], label='International '+str(int(year) - 1))
            
            else:
                # Create stacked bar chart for passengers
                plt.figure(figsize=(6, 5))
                plt.bar(data['Month'], data['DOMESTIC PASSENGERS'], label='Domestic '+str(year))
                plt.bar(data['Month'], data['INTERNATIONAL PASSENGERS'], 
                        bottom=data['DOMESTIC PASSENGERS'], label='International '+str(year))
            
            # Customize the plot
            plt.title("Domestic and International Passengers ("+str(int(year) - 1)+" and "+str(year)+")")
            plt.xlabel("Month")
            plt.ylabel("Number of Passengers (millions)")
            plt.legend(loc='lower right')
            plt.show()

            # Side-by-side bar chart to compare current year with previous year
            if not dataHistory.empty:  # Check if there is data for the previous year
                plt.figure(figsize=(10, 6))
                width = 0.4  # Bar width
                x = range(len(data['Month']))
                # Plot current year data
                plt.bar(data['Month'], data['DOMESTIC FLIGHTS'], width = width, label='Domestic '+str(year))
                plt.bar(data['Month'], data['INTERNATIONAL FLIGHTS'], width = width, 
                        bottom=data['DOMESTIC FLIGHTS'], label='International '+str(year))
                
                # Offset x-axis for the previous year
                x_history = [i - width for i in x]
                
                # Plot previous year data
                plt.bar(x_history, dataHistory['DOMESTIC FLIGHTS'], width = width, label='Domestic '+str(int(year) - 1))
                plt.bar(x_history, dataHistory['INTERNATIONAL FLIGHTS'], width = width, 
                        bottom=dataHistory['DOMESTIC FLIGHTS'], label='International '+str(int(year) - 1))
            else:
                # Create stacked bar chart for passengers
                plt.figure(figsize=(6, 5))
                plt.bar(data['Month'], data['DOMESTIC FLIGHTS'], label='Domestic '+str(year))
                plt.bar(data['Month'], data['INTERNATIONAL FLIGHTS'], 
                        bottom=data['DOMESTIC FLIGHTS'], label='International '+str(year))
                
            # Customize the plot
            plt.title("Domestic and International Flights ("+str(int(year) - 1)+" and "+str(year)+")")
            plt.xlabel("Month")
            plt.ylabel("Number of Flights")
            plt.legend(loc='lower right')
            plt.show()

            # Calculate total domestic and international passengers for the selected year
            totalDomestic = data['DOMESTIC PASSENGERS'].sum()
            totalInternational = data['INTERNATIONAL PASSENGERS'].sum()
            
            # Create data for the pie chart
            categories = ['Domestic Passengers', 'International Passengers']
            sizes = [totalDomestic, totalInternational]
            plt.figure(figsize=(5, 5))
            plt.pie(sizes, labels=categories, autopct='%1.2f%%')
            plt.title("Proportion of Domestic vs International Passengers ("+str(year)+")")
            plt.show()
            
        elif choice == '2':
            return
        else:
            print("Invalid option. Please try again.")

def meeting_space():
    # read csv files and load into dataframe
    df2 = pd.read_csv('Meeting_Space.csv')
    # Change the display options
    pd.set_option('display.max_rows', 500)
    pd.set_option('display.max_columns', 15)
    
    #Change column type
    df2 = df2.drop('Property Count', axis=1)
    df2.index.set_names('Property', inplace=True)
    # Add columns to datafram
    df2['City'] = 'Las Vegas'
    # Add a state column with value set to NV
    df2['State'] = 'NV'
    # convert dataframe columns to integer and address comma seperator
    df2['Hotel Room Inventory'] = df2['Hotel Room Inventory'].str.replace(',', '').astype(int)
    df2['Exhibit Meeting Area'] = df2['Exhibit Meeting Area'].str.replace(',', '').astype(int)
    df2['Hotel Room Inventory'] = df2['Hotel Room Inventory'].fillna('0')
    df2['Exhibit Meeting Area'] = df2['Exhibit Meeting Area'].fillna('0')  
    
    while True:
        print("\n-- Meeting Space Inventory --")
        print("1. Meeting Space Calculator and Recommendation ")
        print("2. Return to Main Menu")
        choice = input("Select an option: ")
    
        if choice == '1':

            # Meeting Space Calculator and Recommendation Engine for Las Vegas Events
            print('Meeting Space Calculator and Recommendation Engine for Las Vegas Events\n')
                
            # Function to get a valid integer input
            def get_valid_integer(prompt):
                while True:
                    user_input = input(prompt)
                    # Check if the input is a digit
                    if user_input.isdigit() and int(user_input) > 0:  
                        return int(user_input)  # Convert to integer and return
                    else:
                        print("Invalid input. Please enter a valid number.")
                
            # Get number of attendees
            attendees = get_valid_integer('Please enter the estimated number of attendees at your event: ')
                
            # Get number of hotel rooms required
            rooms = get_valid_integer('Please enter the number of hotel rooms required for your event: ')
                
            # Get minimum space required
            space_min = get_valid_integer('Please enter the minimum space required (sq. ft) for your meeting: ')
                
            # Get maximum space required
            space_max = get_valid_integer('Please enter the maximum space required (sq. ft) for your meeting: ')
                
            # Display the inputs back to the user
            print("\nEvent Details:")
            print(f"Estimated Attendees: {attendees}")
            print(f"Hotel Rooms Required: {rooms}")
            print(f"Meeting Space Required: {space_min} - {space_max} sq. ft")
                
            # Create boolean mask with user input
            mask_area = (df2['Exhibit Meeting Area'] >= int(space_min)) & (df2['Exhibit Meeting Area'] <= int(space_max))
            mask_room = (df2['Hotel Room Inventory'] >= int(rooms))
                
            # Show results of boolean masking on user input
            df3 = df2.loc[mask_area]
            df4 = df3.loc[mask_room]
            if df4.empty:
                print("\nNo venues match your criteria.")
            else:
                print('----------------------- Vegas IQ ---------------------------\n')
                print('The following meeting venues match your search requirements\n')
                print('Hotel rooms required:                  '+str(rooms))
                print('Exhibit space min (sq.ft.) required:   '+str(space_min))
                print('Exhibit space max (sq.ft.) required:   '+str(space_max))
                df_final = df4[['Property', 'City', 'Exhibit Meeting Area', 'Hotel Room Inventory']]
                print(df_final.to_markdown(index=False))
        elif choice == '2':
            return
        else:
            print("Invalid option. Please try again.")


def jorgeFunction1():
    print("User Management selected. [Placeholder Function]")
    input("Press Enter to return to the main menu...")

def main():
    """Main program loop for VegasIQ."""
    while True:
        choice = main_menu()
        if choice == 1:
            Events_Room_space()
        elif choice == 2:
            olaFunction1()
        elif choice == 3:
            airport_menu()
        elif choice == 4:
            meeting_space()
        elif choice == 5:
            action5Function()
        elif choice == 6:
            clear_screen()
            print("Thank you for using VegasIQ!")
            break
        else:
            print("Invalid choice. Please try again.")
            input("Press Enter to return to the main menu...")

if __name__ == "__main__":
    main()


Welcome to VegasIQ
1. Las Vegas Events and Rooms Database
2. Las Vegas Weather Data
3. Las Vegas Airport Statistics
4. Clark County Meeting Space Inventory
5. User Management
6. System Exit

-------------   Events Database   ------------------
 Las Vegas Travel and Convention Intelligence
 ---------------------------------------------
1. Event Data
2. Event and Accommodation Report
3. Peak Attendance by Week
4. Return to Main Menu

-- Events Database --
1. Filter by Month
2. Filter by Year
3. Filter by Location (Venue)
4. View All Events
5. Return to Main Menu
Venue                                              Event                                                        Start Date   End Date     Est Attendees  
Las Vegas Convention Center                        Consumer Technology Association (CTA) - CES 2024             2024-01-07   2024-01-10   70000          
Mandalay Bay Convention Center, Resort and Casino  The PPAI Expo 2024                                           2024-01-14   

NameError: name 'olaFunction1' is not defined

## 7
